# 02 — Data cleaning

Builds a validated trading calendar, clean adjusted prices, excess returns and quality-control flags.

Reusable implementation and configuration: `src/factor_lens/cleaning.py`.


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "src" / "factor_lens").exists():
    project_root = project_root.parent
if not (project_root / "src" / "factor_lens").exists():
    raise RuntimeError("Open this notebook from the repository root or a subfolder.")
sys.path.insert(0, str(project_root / "src"))


In [2]:
from factor_lens import cleaning as pipeline

clean = pipeline.get_clean()
flags = clean['flags']
universe = clean['universe_clean']
display(clean['gates'])


Cleaning from raw ...
Blocks: A (US stocks) 526 | B (US ETFs) 77 | C (macro) 16 | D (quarantined) 2
Master calendar: 2702 trading days (2016-01-04 -> 2026-10-01); US-ticker observations on non-trading dates dropped: 0
SPY vs ^GSPC (ex-dividend days of SPY excluded): 10 largest |return differences|:
Date
2020-03-16    0.01042
2025-04-09    0.00987
2025-04-10    0.00921
2020-03-13    0.00738
2026-06-26    0.00676
2020-03-17    0.00596
2026-06-29    0.00472
2020-02-28    0.00404
2020-03-26    0.00402
2020-03-27    0.00390

=== FLAGS BY RULE / LABEL / ACTION ===
rule                       label                                                                                  action              
adj_div_mismatch_large     Yahoo proportional adjustment vs cash dividend (regular-size) - used Close+Dividend    use_close_div_return       1
                           unresolved Adj vs Close+Dividend conflict (special dividend/spin-off) - return masked  nan_return                 5
adj_div_mismat

,gate,value,thresholds,status,n_obs
0,SPY vs ^GSPC (non-ex-div days),0.99844,">=0.998 OK, >=0.995 WARN, >=0.98 FAIL",OK,2658
1,SPY - ^GSPC mean spread (annualised),0.01570,0.5%..3% (dividend yield),OK,2658
2,QQQ vs ^NDX (non-ex-div days),0.99873,">=0.998 OK, >=0.995 WARN, >=0.98 FAIL",OK,2658
3,IWM vs ^RUT (non-ex-div days),0.99642,">=0.995 OK, >=0.99 WARN, >=0.97 FAIL",OK,2658
4,SMH vs ^SOX (non-ex-div days),0.99068,">=0.985 OK, >=0.97 WARN, >=0.93 FAIL",OK,2691
5,EW avg of S&P stocks vs RSP,0.99225,">=0.985 OK, >=0.97 WARN, >=0.90 FAIL",OK,2701


## Follow-up review

Inspect the saved `flags` and `universe` tables when a validation gate is warning or bad.


In [3]:
display(flags[flags['rule'].isin(['outlier', 'adj_div_mismatch_large', 'stale_price'])].head(40))


,date,ticker,rule,label,value,action
0,2017-07-05,BKR,adj_div_mismatch_large,unresolved Adj vs Close+Dividend conflict (spe...,-0.022124,nan_return
1,2016-07-05,DHR,adj_div_mismatch_large,unresolved Adj vs Close+Dividend conflict (spe...,0.218627,nan_return
2,2020-02-03,GEN,adj_div_mismatch_large,unresolved Adj vs Close+Dividend conflict (spe...,0.018772,nan_return
3,2018-07-10,KDP,adj_div_mismatch_large,unresolved Adj vs Close+Dividend conflict (spe...,0.096077,nan_return
4,2020-03-09,OXY,adj_div_mismatch_large,Yahoo proportional adjustment vs cash dividend...,-0.015298,use_close_div_return
5,2021-12-14,WULF,adj_div_mismatch_large,unresolved Adj vs Close+Dividend conflict (spe...,-0.043891,nan_return
478,2019-05-15,A,outlier,genuine_event,-0.110117,flag_only
479,2026-05-28,A,outlier,genuine_event,0.168681,flag_only
480,2020-07-31,AAPL,outlier,genuine_event,0.104689,flag_only
481,2018-01-26,ABBV,outlier,genuine_event,0.137673,flag_only
